# Slurm EDA - Analysis (Rough)

In this Jupyter Notebook, we continue the EDA started in the previous notebooks, analysing the data by creating plots. 

## Preparing the Notebook

### Importing the necessary libraries

We will first import all of the necessary libraries. 

In [24]:
import pandas as pd
import numpy as np
import plotly 
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp

### Importing the data

We have saved the relevant data in two .csv files which we will now load into two DataFrames. 

In [2]:
# We will now read the .csv file containing the Slurm data for June into a DataFrame
sSlurmDataPath = '../data/dfSacctFinal.csv'
dfSacct = pd.read_csv(sSlurmDataPath, index_col=0, parse_dates=['Start', 'End'], infer_datetime_format=True)

# We will now read the .csv file containing the extended Slurm data for June into a DataFrame. 
sSlurmExtendedPath = '../data/dfSacctExtended.csv'
dfSacctExtended = pd.read_csv(sSlurmExtendedPath, index_col=0, parse_dates=['Start', 'End'], infer_datetime_format=True)

## EDA

### Data Cleaning

We will first create a new column 'State' column without the users that cancelled the job. We will also contain a column that only contains the user ID if the job was cancelled (and None if the job was not cancelled)

In [3]:
# We first create a new column, since we may use the other column in later analysis. 
dfSacct['StateShort'] = dfSacct['State'].copy()

# We now remove the user from the 'CANCELLED by ...' values
dfSacct['StateShort'] = dfSacct['StateShort'].where(~(dfSacct['StateShort'].str.contains('CANCELLED')), 'CANCELLED')

# We will now create another column that will contain the user ID for cancelled jobs. 
dfSacct['CancelledID'] = dfSacct['State'].copy()

# We will now create a function to replace the value with the user's ID
def replaceID(row):
    sState = row.State

    if 'CANCELLED' not in sState:
        return None
    
    sUserID = sState.split()[-1]

    return sUserID

# We will now apply our function to every row of the DataFrame
dfSacct['CancelledID'] = dfSacct.apply(lambda row : replaceID(row), axis=1)

We will now create separate DataFrames containing the exclusive and shared jobs respectively.

In [5]:
# We will first create a DataFrame containing only the exclusive jobs. 
dfSacctExclusive = dfSacct[dfSacct['Exclusive']]

# We will now create a DataFrame containing only the shared jobs. 
dfSacctShared = dfSacct[~dfSacct['Exclusive']]

### Analysis

We will first calculate some summary statistics.

In [6]:
# Below we calculate the percentage of total jobs which is exclusive (based on job count)
iTotalExclusive = sum(dfSacct['Exclusive'])
iPercentageExclusiveCount = iTotalExclusive/len(dfSacct) * 100

# Below we calculate the percentage of total runtime which is exclusive. 
iTotalRuntime = sum(dfSacct['CPUTimeRAW'])
iExclusiveRuntime = sum(dfSacct[dfSacct['Exclusive']]['CPUTimeRAW'])
iPercentageExclusiveRuntime = iExclusiveRuntime/iTotalRuntime * 100

# Below we calculate the percentage of shared jobs which are only shared with jobs from the same user.
# We will first find the percentage by job count. 
iTotalSameUser = sum(dfSacct['SharedSameUser'])
iTotalShared = len(dfSacct) - sum(dfSacct['Exclusive'])
iPercentageSameUser = iTotalSameUser/iTotalShared * 100

# We will now find the percentage by runtime
iTotalSameUserRuntime = sum(dfSacct[dfSacct['SharedSameUser']]['CPUTimeRAW'])
iTotalSharedRuntime = sum(dfSacct[dfSacct['Exclusive'] == False]['CPUTimeRAW'])
iPercentageSameUserRuntime = iTotalSameUserRuntime/iTotalSharedRuntime * 100

# We will now calculate the percentage of exclusive jobs which are exclusive by CPU count or Overlap
iTotalExclusiveCPU = sum(dfSacct['ExclusiveCPU'])
iTotalExclusiveOverlap = sum(dfSacct['ExclusiveOverlapping'])

iPercentageExclusiveCPU = iTotalExclusiveCPU/iTotalExclusive * 100
iPercentageExclusiveOverlap = iTotalExclusiveOverlap/iTotalExclusive * 100

print(iPercentageExclusiveCount, '% of jobs are exclusive.')
print(iPercentageExclusiveRuntime, '% of total runtime is made up of exclusive jobs.')
print('\n')
print(iPercentageExclusiveCPU, '% of exclusive jobs run accross whole nodes.')
print(iPercentageExclusiveOverlap, '% of exclusive jobs do not use a whole node.')
print('\n')
print(iPercentageSameUser, '% of shared jobs are run by the same user (by job count).')
print(iPercentageSameUserRuntime, '% of shared jobs are run by the same user (by runtime).')


We will start by outputting the first 5 rows of the DataFrame to get an idea of the data that we are working with. 

In [7]:
dfSacct.head()

We will now calculate the 5 number summary for both DataFrames

In [9]:
dfSacctSummary = dfSacct.describe()
dfSacctExtendedSummary = dfSacctExtended.describe()

We will now remove all outliers from the DataFrames

In [10]:
# We will filter the two DataFrames to remove outliers. 

def removeOutliers(df):
    """This function will remove any outliers from a given DataFrame"""

    # We first create the corresponding summary DataFrame
    dfSummary = df.describe()

    # We first add a new row to the summary DataFrame including the IQR for each column. 
    dfSummary.loc['IQR'] = dfSummary.loc['75%'] - dfSummary.loc['25%']

    # We then add two more new rows to the summary DataFrame containing the outlier bounds for each column. 
    dfSummary.loc['LowerBound'] = dfSummary.loc['25%'] - 1.5 * dfSummary.loc['IQR']
    dfSummary.loc['UpperBound'] = dfSummary.loc['75%'] + 1.5 * dfSummary.loc['IQR']

    # We now loop through each column of the summary DataFrame creating a boolean mask to remove the outliers for each 
    # column. When we have two boolean masks we combine them into one. 
    lMasks = []

    for sColumnName in dfSummary.columns:
        lMasks.append((df[sColumnName] >= dfSummary.loc['LowerBound', sColumnName]) & (df[sColumnName] <= dfSummary.loc['UpperBound', sColumnName]))
        if len(lMasks) > 1:
            lMasks = [(lMasks[0] & lMasks[1])]

    if len(lMasks) > 1:
            lMasks = [(lMasks[0] & lMasks[1])]

    # We return the final DataFrame containing no outliers. 
    return df[lMasks[0]]

dfSacctClean = removeOutliers(dfSacct)
dfExtendedClean = removeOutliers(dfSacctExtended)

We will now plot some graphs to visualize the data that we are working with. 

In [11]:
# Below we plot a frequency histogram of the allocated number of nodes for a job (with no outliers). 

fig = px.histogram(dfSacctClean, x='AllocNodes')

fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of the Number of Allocated Nodes for a Job (No Outliers)', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='Number of Allocated Nodes')
                 )
fig.show()

In [12]:
# Below we plot a frequency histogram of the allocated number of nodes for a job (with outliers). 

fig = px.histogram(dfSacct, x='AllocNodes')
fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of the Number of Allocated Nodes for a Job (Outliers Present)', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='Number of Allocated Nodes')
                 )
fig.show()

In [13]:
# Below we plot a frequency histogram of the allocated number of nodes for a job (with outliers) on a log scale. 

fig = px.histogram(dfSacct, x='AllocNodes', log_x=True)
fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of the Number of Allocated Nodes for a Job (Outliers Present)', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='log Number of Allocated Nodes')
                 )
fig.show()

In [14]:
# Below we plot a histogram of the number of allocated nodes for a job (including the outliers) 
# with the frequency as the CPU time.

fig = px.histogram(dfSacct, x='AllocNodes', y='CPUTimeRAW')
fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of the Number of Allocated Nodes for a Job by CPU Time (Outliers Present)', x=0.5),
                  yaxis_title=dict(text='CPU Time (seconds)'),
                  xaxis_title=dict(text='Number of Allocated Nodes')
                 )
fig.show()

In [15]:
# Below we plot two histograms of the number of allocated nodes for a job (including the outliers) 
# with the frequency as the CPU time for shared and exclusive jobs respectively.

fig = px.histogram(dfSacct, x='AllocNodes', y='CPUTimeRAW', color='Exclusive', barmode='overlay')
fig.update_layout(showlegend=True,
                  title=dict(text='Histogram of the Number of Allocated Nodes for a Job by CPU Time (Outliers Present)', x=0.5),
                  yaxis_title=dict(text='CPU Time (seconds)'),
                  xaxis_title=dict(text='Number of Allocated Nodes')
                 )
fig.show()

In [16]:
# Below we plot a frequency histogram of the number of allocated CPUs for a job (with no outliers)

fig = px.histogram(dfSacctClean, x='AllocCPUS')
fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of the Number of Allocated CPUs for a Job (No Outliers)', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='Number of Allocated CPUs')
                 )
fig.show()

In [17]:
# Below we plot a frequency histogram of the number of allocated CPUs for a job (with outliers)

fig = px.histogram(dfSacct, x='AllocCPUS')
fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of the Number of Allocated CPUs for a Job (Outliers Present)', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='Number of Allocated CPUs')
                 )
fig.show()

In [18]:
# Below we plot a frequency histogram of the number of allocated CPUs for a job (with outliers) on a log scale.

fig = px.histogram(dfSacct, x='AllocCPUS', log_x=True)

fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of the Number of Allocated CPUs for a Job (Outliers Present)', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='log Number of Allocated CPUs'),
                 )

fig.show()

The 6 plots above illustrate the importance of keeping the outliers in our DataFrames. As you can see by comparing the bar charts without outliers to those with outliers, we lose a lot of information by removing the outliers. Some important examples to note are: 

    - All number of nodes apart from 1 node are outliers (statistically) however we are interested in jobs running on multiple nodes. 
    - The second highest peak for the number of allocated CPUs (32 CPUs) is an outlier. However we may be interested in why there is a peak here. 

However, when we use a log scale on the x axis, we can see all of the data points while keeping the outliers in the data.


We will now plot a frequency histogram of CPU time of a job. 

In [19]:
fig = px.histogram(dfSacct, x='CPUTimeRAW')
fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of the CPU time for a Job (Outliers Present)', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='Number of Allocated Nodes')
                 )
fig.show()

In [20]:
fig = px.histogram(dfSacctClean, x='CPUTimeRAW')
fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of the CPU time for a Job (No Outliers)', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='Runtime')
                 )
fig.show()

On the other hand, as you can see from the two graphs above, it can be important to remove outliers as it allows us to visualise the distribution of the majority of the data. 

## User job analysis

We will now analyse users' job data to get an idea of how to display their carbon footprint data back to them.

In [21]:
# We will begin by creating a list of DataFrames, each one containing only the jobs for one user. 
lUserDFs = []

for user in dfSacct['User'].unique():
    bUserMask = dfSacct['User'] == user
    lUserDFs.append(dfSacct[bUserMask])

In [22]:
# We will now create a list of the lengths of each users' DataFrame (how many jobs they have)
lUserJobCount = [len(lUserDF) for lUserDF in lUserDFs]

In [27]:
# We will now find the 5 number summary of the job counts. 

iMax = np.max(lUserJobCount)
iMin = np.min(lUserJobCount)
iMedian = np.median(lUserJobCount)
iMean = np.mean(lUserJobCount)
iStd = np.std(lUserJobCount)

print('Max:', iMax)
print('Min:', iMin)
print('Median:', iMedian)
print('Mean:', iMean)
print('Standard deviation:', iStd)

We will now plot a histogram of the job count data.

In [29]:
fig = px.histogram(lUserJobCount)
fig.update_layout(showlegend=False,
                  title=dict(text='Number of Jobs Per User', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='Jobs Per User')
                 )
fig.show()

We will now show only job counts less than 1000.

In [32]:
lUserJobCountFiltered = [iCount for iCount in lUserJobCount if iCount < 1000]

In [33]:
fig = px.histogram(lUserJobCountFiltered)
fig.update_layout(showlegend=False,
                  title=dict(text='Number of Jobs Per User', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='Jobs Per User')
                 )
fig.show()

In [36]:
lUserJobCounts100 = [iCount for iCount in lUserJobCount if iCount <= 100]

i100JobsPercentage = len(lUserJobCounts100)/len(lUserJobCount) * 100

print(f'{i100JobsPercentage}% of users have 100 jobs or less.')